In [1]:
!pip install pymupdf

   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
   ---------------------------------------- 0.1/18.4 MB 1.5 MB/s eta 0:00:13
   - -------------------------------------- 0.9/18.4 MB 7.9 MB/s eta 0:00:03
   --- ------------------------------------ 1.4/18.4 MB 9.0 MB/s eta 0:00:02
   ---- ----------------------------------- 2.0/18.4 MB 9.6 MB/s eta 0:00:02
   ----- ---------------------------------- 2.4/18.4 MB 10.1 MB/s eta 0:00:02
   ------ --------------------------------- 2.9/18.4 MB 10.2 MB/s eta 0:00:02
   ------ --------------------------------- 3.1/18.4 MB 9.8 MB/s eta 0:00:02
   -------- ------------------------------- 3.8/18.4 MB 10.0 MB/s eta 0:00:02
   --------- ------------------------------ 4.4/18.4 MB 10.4 MB/s eta 0:00:02
   ---------- ----------------------------- 4.8/18.4 MB 10.6 MB/s eta 0:00:02
   ----------- ---------------------------- 5.3/18.4 MB 10.7 MB/s eta 0:00:02
   ----


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import re
import fitz  # PyMuPDF
import pandas as pd
from pathlib import Path
from typing import List, Tuple

# ---------- ICD code patterns ----------
# Accept 3-char (A00) and dotted subcodes (A00.0, A18.0, A15.0X etc.)
RE_CODE_ONLY = re.compile(r"^[A-Z][0-9]{2}(?:\.[0-9A-Z]+)?$")  # line contains only the code
RE_CODE_DESC_INLINE = re.compile(r"\b([A-Z][0-9]{2}(?:\.[0-9A-Z]+)?)\b\s{1,}(.+)$")
RE_RANGE = re.compile(r"[A-Z][0-9]{2}\s*[–—-]\s*[A-Z][0-9]{2}")  # A00–B99 header ranges

def parse_page_ranges(pages_arg: str, max_pages: int) -> List[Tuple[int, int]]:
    """Parse '5-28,40-45,60' (1-based) -> list of (start,end) 0-based inclusive."""
    ranges = []
    for chunk in pages_arg.split(","):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "-" in chunk:
            a, b = chunk.split("-", 1)
            start = int(a); end = int(b)
        else:
            start = end = int(chunk)
        if start < 1 or end < 1 or start > end:
            raise ValueError(f"Invalid range '{chunk}'. Use 1-based inclusive ranges like 5-28.")
        if start > max_pages or end > max_pages:
            raise ValueError(f"Range '{chunk}' exceeds document length ({max_pages} pages).")
        ranges.append((start - 1, end - 1))
    return ranges

def normalize_ranges(ranges: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Merge overlapping/adjacent ranges."""
    if not ranges:
        return []
    ranges.sort()
    merged = [ranges[0]]
    for s, e in ranges[1:]:
        ms, me = merged[-1]
        if s <= me + 1:
            merged[-1] = (ms, max(me, e))
        else:
            merged.append((s, e))
    return merged

def clean_line(s: str) -> str:
    if s is None:
        return ""
    # Normalize whitespace & dashes
    s = s.replace("\u00a0", " ").replace("\u2013", "-").replace("\u2014", "-")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def is_heading_or_range(line: str) -> bool:
    if not line:
        return True
    low = line.lower()
    if low.startswith("chapter"):
        return True
    if RE_RANGE.search(line):
        return True
    return False

def iter_lines_from_ranges(pdf_path: str, ranges_0based: List[Tuple[int, int]]):
    with fitz.open(pdf_path) as doc:
        for (s, e) in ranges_0based:
            for page_num in range(s, e + 1):
                page = doc[page_num]
                text = page.get_text("text")
                for ln in text.splitlines():
                    yield clean_line(ln)

In [9]:
def parse_icd_all_codes(lines_iter):
    """
    Capture every ICD code (3-char or dotted) with exactly one description:
    - Inline: 'A00  Cholera' -> ('A00', 'Cholera')
    - Stacked: 
         A00
         Cholera
    - Stop at the next line (no multi-line concatenation).
    - Ignore chapter headings and range lines.
    """
    results = []
    pending_code = None
    waiting_for_desc = False

    def emit(code, desc):
        desc = (desc or "").strip()
        desc = re.sub(r"\s+", " ", desc)
        results.append({"code": code, "description": desc})

    for line in lines_iter:
        if not line:
            # If waiting for desc, skip blanks until we get one or something else terminates it
            continue

        if is_heading_or_range(line):
            if waiting_for_desc and pending_code:
                emit(pending_code, "")
                pending_code, waiting_for_desc = None, False
            continue

        # Inline case first: CODE  Description
        m = RE_CODE_DESC_INLINE.match(line)
        if m:
            if waiting_for_desc and pending_code:   # flush orphaned
                emit(pending_code, "")
            code = m.group(1)
            desc = m.group(2)
            emit(code, desc)
            pending_code, waiting_for_desc = None, False
            continue

        # Code-only line
        if RE_CODE_ONLY.match(line):
            if waiting_for_desc and pending_code:
                # previous code had no desc line
                emit(pending_code, "")
            pending_code = line
            waiting_for_desc = True
            continue

        # If we are expecting exactly one desc line for a code
        if waiting_for_desc and pending_code:
            # If the "desc" line itself starts a new code or is heading/range, previous had no desc
            if RE_CODE_ONLY.match(line) or is_heading_or_range(line):
                emit(pending_code, "")
                pending_code, waiting_for_desc = None, False
                # Reprocess this line quickly (handles back-to-back codes)
                if RE_CODE_ONLY.match(line):
                    pending_code = line
                    waiting_for_desc = True
                # (if it was heading/range, we just continue)
            else:
                emit(pending_code, line)
                pending_code, waiting_for_desc = None, False
            continue

        # Otherwise unrelated text—ignore
        continue

    # Tail: if the last code had no desc
    if waiting_for_desc and pending_code:
        emit(pending_code, "")

    return results

In [10]:
def extract_all_icd_codes_to_files(
    pdf_path: str,
    pages: str = None,     # e.g., "5-28,40-45,60"
    start: int = None,     # 1-based inclusive
    end: int = None,       # 1-based inclusive
    out_csv: str = "icd10_codes_all_levels.csv",
    out_xlsx: str = "icd10_codes_all_levels.xlsx",
    return_df: bool = True
):
    pdf_path = str(pdf_path)
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    with fitz.open(pdf_path) as doc:
        total_pages = len(doc)

    if pages:
        ranges = parse_page_ranges(pages, total_pages)
    else:
        if start is None or end is None:
            raise ValueError("Provide either pages='a-b[,c-d]' OR both start and end.")
        if start < 1 or end < 1 or start > end:
            raise ValueError("Invalid start/end values. They must be 1-based and start <= end.")
        if start > total_pages or end > total_pages:
            raise ValueError(f"start/end exceed document length ({total_pages} pages).")
        ranges = [(start - 1, end - 1)]

    ranges = normalize_ranges(ranges)
    human_ranges = ", ".join([f"{s+1}-{e+1}" if s != e else f"{s+1}" for s, e in ranges])
    print(f"Processing page range(s): {human_ranges} (1-based). Total pages: {total_pages}")

    lines_iter = iter_lines_from_ranges(pdf_path, ranges)
    records = parse_icd_all_codes(lines_iter)

    # Optional: de-duplicate (keep first occurrence)
    seen = set()
    dedup = []
    for r in records:
        if r["code"] not in seen:
            seen.add(r["code"])
            dedup.append(r)

    df = pd.DataFrame(dedup, columns=["code", "description"])

    # Sort alphanumerically (letter then number; dotted subcodes after parents)
    def sort_key(c):
        # Split like "A00.1" -> ("A", 0, 0, "1")
        base = c.split(".")
        letter = base[0][0]
        num = int(base[0][1:])
        sub = base[1] if len(base) > 1 else ""
        # ensure dotted codes come after the 3-char parent
        dotted_flag = 1 if sub else 0
        return (letter, num, dotted_flag, sub)
    if not df.empty:
        df = df.sort_values(by="code", key=lambda s: s.map(sort_key)).reset_index(drop=True)

    # Save
    Path(out_csv).write_text(df.to_csv(index=False), encoding="utf-8")
    df.to_excel(out_xlsx, index=False, engine="openpyxl")
    print(f"Extracted {len(df)} codes (all levels).")
    print(f"CSV  -> {Path(out_csv).resolve()}")
    print(f"XLSX -> {Path(out_xlsx).resolve()}")

    return df if return_df else None


In [12]:
PDF_PATH = r"C:\Users\UNegi\Downloads\icd-10-am_chronicle_-_eleventh_edition.pdf"  # point to your file

# EITHER use pages string:
PAGES = "2-1271"       # or like "5-28,40-45,60"
START, END = None, None

# OR use start/end:
# PAGES = None
# START, END = 5, 28

df_all = extract_all_icd_codes_to_files(
    pdf_path=PDF_PATH,
    pages=PAGES,
    start=START,
    end=END,
    out_csv="icd10_codes_all_levels.csv",
    out_xlsx="icd10_codes_all_levels.xlsx",
    return_df=True
)

df_all.head(12)


Processing page range(s): 2-1271 (1-based). Total pages: 2797
Extracted 40676 codes (all levels).
CSV  -> C:\Users\UNegi\Documents\Project\makethon\icd10_codes_all_levels.csv
XLSX -> C:\Users\UNegi\Documents\Project\makethon\icd10_codes_all_levels.xlsx


,code,description
0,A00,Cholera
1,A00.0,"Cholera due to Vibrio cholerae 01, biovar chol..."
2,A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor"
3,A00.9,"Cholera, unspecified"
4,A01,Typhoid and paratyphoid fevers
5,A01.0,Typhoid fever
6,A01.1,Paratyphoid fever A
7,A01.2,Paratyphoid fever B
8,A01.3,Paratyphoid fever C
9,A01.4,"Paratyphoid fever, unspecified"


In [13]:
df_top = df_all[df_all["code"].str.fullmatch(r"[A-Z][0-9]{2}")]
df_top.to_csv("icd10_codes_top_level_only.csv", index=False)
df_top.head()

,code,description
0,A00,Cholera
4,A01,Typhoid and paratyphoid fevers
10,A02,Other salmonella infections
16,A03,Shigellosis
23,A04,Other bacterial intestinal infection


## NEW

In [14]:
import re
import fitz  # PyMuPDF
import pandas as pd
from pathlib import Path
from typing import List, Tuple

In [15]:
# ---------- ICD code patterns ----------
# Accept 3-char (A00) and dotted subcodes (A00.0, A18.0, A15.0X etc.)
RE_CODE_ONLY = re.compile(r"^[A-Z][0-9]{2}(?:\.[0-9A-Z]+)?$")
# Inline: CODE  Description
RE_CODE_DESC_INLINE = re.compile(r"\b([A-Z][0-9]{2}(?:\.[0-9A-Z]+)?)\b\s{1,}(.+)$")
# Range headers like "A15–A19"
RE_RANGE = re.compile(r"[A-Z][0-9]{2}\s*[–—-]\s*[A-Z][0-9]{2}")

def parse_page_ranges(pages_arg: str, max_pages: int) -> List[Tuple[int, int]]:
    """Parse '5-28,40-45,60' (1-based) -> list of (start,end) 0-based inclusive."""
    ranges = []
    for chunk in pages_arg.split(","):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "-" in chunk:
            a, b = chunk.split("-", 1)
            start = int(a); end = int(b)
        else:
            start = end = int(chunk)

        if start < 1 or end < 1 or start > end:
            raise ValueError(f"Invalid range '{chunk}'. Use 1-based inclusive ranges like 5-28.")
        if start > max_pages or end > max_pages:
            raise ValueError(f"Range '{chunk}' exceeds document length ({max_pages} pages).")
        ranges.append((start - 1, end - 1))
    return ranges

def normalize_ranges(ranges: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Merge overlapping/adjacent ranges."""
    if not ranges:
        return []
    ranges.sort()
    merged = [ranges[0]]
    for s, e in ranges[1:]:
        ms, me = merged[-1]
        if s <= me + 1:
            merged[-1] = (ms, max(me, e))
        else:
            merged.append((s, e))
    return merged

def clean_line(s: str) -> str:
    """Normalize whitespace and dash variants."""
    if s is None:
        return ""
    s = s.replace("\u00a0", " ").replace("\u2013", "-").replace("\u2014", "-")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def is_heading_or_range(line: str) -> bool:
    """Detect headings like 'Chapter ...' or code ranges 'A00-B99'."""
    if not line:
        return True
    low = line.lower()
    if low.startswith("chapter"):
        return True
    if RE_RANGE.search(line):
        return True
    return False

def iter_lines_from_ranges(pdf_path: str, ranges_0based: List[Tuple[int, int]]):
    """Yield cleaned lines from the requested page ranges (in reading order)."""
    with fitz.open(pdf_path) as doc:
        for (s, e) in ranges_0based:
            for page_num in range(s, e + 1):
                page = doc[page_num]
                text = page.get_text("text")
                for ln in text.splitlines():
                    yield clean_line(ln)

In [16]:
def should_stop_desc(line: str) -> bool:
    """
    Stop description when:
      - blank line
      - new code line
      - chapter/range header
    """
    if not line:
        return True
    if is_heading_or_range(line):
        return True
    if RE_CODE_ONLY.match(line):
        return True
    return False

def merge_hyphenation(prev: str, curr: str) -> str:
    """
    If previous chunk ends with a hyphen (soft wrap), join without space.
    Otherwise, join with a space. Normalize spaces.
    """
    if not prev:
        joined = curr or ""
    else:
        if prev.endswith("-"):
            joined = prev[:-1] + (curr.lstrip() if curr else "")
        else:
            joined = prev + " " + (curr.lstrip() if curr else "")
    return re.sub(r"\s+", " ", joined).strip()


In [17]:
def parse_icd_all_codes_with_wraps(lines_iter, max_cont_lines: int = 10):
    """
    Capture EVERY ICD code with multi-line description:
    - Inline: 'A15  Tuberculosis' -> take inline desc
    - Stacked: 'A15' then one or more description lines
    - Append continuation lines until next code / range / heading / blank
    - Fix soft hyphenation across line breaks
    - Guard with max_cont_lines to prevent runaway merges
    """
    results = []
    pending_code = None
    capturing = False
    desc_parts = []
    cont_count = 0

    def emit(code, parts):
        if not code:
            return
        desc = ""
        for p in parts:
            if p:
                desc = merge_hyphenation(desc, p) if desc else p
        desc = re.sub(r"\s+", " ", (desc or "")).strip()
        results.append({"code": code, "description": desc})

    for raw in lines_iter:
        line = clean_line(raw)

        # 1) Inline "CODE  Description"
        m = RE_CODE_DESC_INLINE.match(line) if line else None
        if m:
            if capturing and pending_code:
                emit(pending_code, desc_parts)
            pending_code = m.group(1)
            first_desc = m.group(2).strip() if m.group(2) else ""
            desc_parts = [first_desc] if first_desc else []
            capturing = True
            cont_count = 0
            continue

        # 2) Code-only line
        if line and RE_CODE_ONLY.match(line):
            if capturing and pending_code:
                emit(pending_code, desc_parts)
            pending_code = line
            desc_parts = []
            capturing = True
            cont_count = 0
            continue

        # 3) Continuations
        if capturing and pending_code:
            if should_stop_desc(line):
                emit(pending_code, desc_parts)
                pending_code, desc_parts, capturing = None, [], False
                cont_count = 0
                # Re-process boundary if it's another code/inline
                if line:
                    m2 = RE_CODE_DESC_INLINE.match(line)
                    if m2:
                        pending_code = m2.group(1)
                        desc_parts = [m2.group(2).strip()] if m2.group(2) else []
                        capturing = True
                        cont_count = 0
                    elif RE_CODE_ONLY.match(line):
                        pending_code = line
                        desc_parts = []
                        capturing = True
                        cont_count = 0
                continue

            if line:
                desc_parts.append(line)
                cont_count += 1
                if cont_count >= max_cont_lines:
                    emit(pending_code, desc_parts)
                    pending_code, desc_parts, capturing = None, [], False
                    cont_count = 0
            continue

        # 4) No active capture -> skip
        continue

    # Tail flush
    if capturing and pending_code:
        emit(pending_code, desc_parts)

    return results

In [18]:
def sort_alphanum_code_series(s: pd.Series) -> pd.Series:
    """
    Sort codes alphanumerically:
      - by letter (A..Z),
      - by numeric part (00..99),
      - dotted subcodes after parent,
      - then by dotted suffix string.
    """
    def key(c: str):
        c = str(c)
        base = c.split(".")
        letter = base[0][0]
        num = int(base[0][1:]) if base[0][1:].isdigit() else 0
        sub = base[1] if len(base) > 1 else ""
        dotted_flag = 1 if sub else 0
        return (letter, num, dotted_flag, sub)
    return s.map(key)

def extract_icd_all_codes_to_files(
    pdf_path: str,
    pages: str = None,     # e.g., "5-28,40-45,60"
    start: int = None,     # 1-based inclusive
    end: int = None,       # 1-based inclusive
    out_csv: str = "icd10_codes_all_levels.csv",
    out_xlsx: str = "icd10_codes_all_levels.xlsx",
    max_cont_lines: int = 10,
    return_df: bool = True
):
    """
    Notebook-friendly runner:
    - Reads selected pages
    - Parses all ICD codes with wrapped descriptions
    - Saves CSV/XLSX
    - Returns DataFrame (optional)
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    with fitz.open(pdf_path) as doc:
        total_pages = len(doc)

    if pages:
        ranges = parse_page_ranges(pages, total_pages)
    else:
        if start is None or end is None:
            raise ValueError("Provide either pages='a-b[,c-d]' OR both start and end.")
        if start < 1 or end < 1 or start > end:
            raise ValueError("Invalid start/end values. They must be 1-based and start <= end.")
        if start > total_pages or end > total_pages:
            raise ValueError(f"start/end exceed document length ({total_pages} pages).")
        ranges = [(start - 1, end - 1)]

    ranges = normalize_ranges(ranges)
    human_ranges = ", ".join([f"{s+1}-{e+1}" if s != e else f"{s+1}" for s, e in ranges])
    print(f"Processing page range(s): {human_ranges} (1-based). Total pages: {total_pages}")

    # Extract & parse
    lines_iter = iter_lines_from_ranges(str(pdf_path), ranges)
    records = parse_icd_all_codes_with_wraps(lines_iter, max_cont_lines=max_cont_lines)

    # Deduplicate by code (keep first seen)
    seen = set()
    rows = []
    for r in records:
        if r["code"] not in seen:
            seen.add(r["code"])
            rows.append(r)

    df = pd.DataFrame(rows, columns=["code", "description"])
    if not df.empty:
        df = df.sort_values(by="code", key=lambda s: sort_alphanum_code_series(s)).reset_index(drop=True)

    # Save
    Path(out_csv).write_text(df.to_csv(index=False), encoding="utf-8")
    df.to_excel(out_xlsx, index=False, engine="openpyxl")

    print(f"Extracted {len(df)} codes (all levels).")
    print(f"CSV  -> {Path(out_csv).resolve()}")
    print(f"XLSX -> {Path(out_xlsx).resolve()}")

    return df if return_df else None

In [19]:
# ==== EDIT THESE ====
PDF_PATH = r"C:\Users\UNegi\Downloads\icd-10-am_chronicle_-_eleventh_edition.pdf"  # path to your PDF (place it next to the notebook or use full path)

# EITHER use pages string:
PAGES = "2-1271"                   # e.g., "5-28,40-45,60"
START, END = None, None

# OR use start/end:
# PAGES = None
# START, END = 5, 28

OUT_CSV = "icd10_codes_all_levels.csv"
OUT_XLSX = "icd10_codes_all_levels.xlsx"

# Adjust if your descriptions wrap a lot
MAX_CONT_LINES = 3
# ====================

df_all = extract_icd_all_codes_to_files(
    pdf_path=PDF_PATH,
    pages=PAGES,
    start=START,
    end=END,
    out_csv=OUT_CSV,
    out_xlsx=OUT_XLSX,
    max_cont_lines=MAX_CONT_LINES,
    return_df=True
)

df_all.head(15)

Processing page range(s): 2-1271 (1-based). Total pages: 2797
Extracted 40676 codes (all levels).
CSV  -> C:\Users\UNegi\Documents\Project\makethon\icd10_codes_all_levels.csv
XLSX -> C:\Users\UNegi\Documents\Project\makethon\icd10_codes_all_levels.xlsx


,code,description
0,A00,Cholera
1,A00.0,"Cholera due to Vibrio cholerae 01, biovar chol..."
2,A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor"
3,A00.9,"Cholera, unspecified"
4,A01,Typhoid and paratyphoid fevers
5,A01.0,Typhoid fever
6,A01.1,Paratyphoid fever A
7,A01.2,Paratyphoid fever B
8,A01.3,Paratyphoid fever C
9,A01.4,"Paratyphoid fever, unspecified"


In [20]:
df_top = df_all[df_all["code"].str.fullmatch(r"[A-Z][0-9]{2}")]
df_top.to_csv("icd10_top_level_only.csv", index=False)
df_top.head(10)

,code,description
0,A00,Cholera
4,A01,Typhoid and paratyphoid fevers
10,A02,Other salmonella infections
16,A03,Shigellosis
23,A04,Other bacterial intestinal infection
34,A05,"Other bacterial food-borne intoxications, not ..."
42,A06,Amoebiasis
53,A07,Other protozoal intestinal diseases
60,A08,Viral and other specified intestinal infections
67,A09,Other gastroenteritis and colitis of infectiou...


In [22]:
df1 = pd.read_excel(r"C:\Users\UNegi\OneDrive - Cognitio Analytics LLC\makethon\icd_10_cms_group.xlsx")

In [25]:
df1.columns

Index(['Code', 'Description', 'Code_Super_Grouper'], dtype='object')

In [23]:
df2 = df_top.copy()

In [24]:
df2.columns

Index(['code', 'description'], dtype='object')

In [27]:
df2.rename(columns={'code': 'code_new'}, inplace=True)

In [28]:
merged_df = df1.merge(
    df2,
    left_on="Code_Super_Grouper",
    right_on="code_new",
    how="left"
)

In [30]:
merged_df.to_excel(r"C:\Users\UNegi\OneDrive - Cognitio Analytics LLC\makethon\icd_10_cms_super_grouper.xlsx")